In [31]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [32]:
load_dotenv()

True

In [33]:
model = ChatOpenAI(model='gpt-4o-mini')

In [34]:
class SentimentSchema(BaseModel):

    sentiment: Literal["positive", "negative"] = Field(description="Sentiment of the review")

In [35]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappoionted", "calm"]=Field(description="The emotional tone expressed by the user")
    urgency: Literal["low", "medium", "high"]=Field(description='How urgent or critical the issue appears to be')

In [36]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

In [37]:
class ReviewState(TypedDict):

    review : str
    sentiment: Literal["positive", "negative"]
    diagnosis : dict
    response: str


In [38]:
# --- Node Functions ---

def findSentiment(state: ReviewState) -> dict:
    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    # Assuming structured_model returns a Pydantic model matching SentimentSchema
    sentiment = structured_model.invoke(prompt).sentiment 
    return {'sentiment': sentiment}

# FIX: Explicitly check if the string equals "positive"
def checkSentiment(state: ReviewState) -> Literal["positiveResponse", "runDiagnosis"]:
    if state.get('sentiment') == "positive":
        return "positiveResponse"
    else:
        return "runDiagnosis"

def positiveResponse(state: ReviewState) -> dict:
    prompt = f"""Write a warm Thank-you message in response to this review:\n\n{state['review']} 
    Also Kindly ask the user to leave feedback to our website"""
    response = model.invoke(prompt).content
    return {'response': response}

def runDiagnosis(state: ReviewState) -> dict:
    prompt = f"""Diagnose this negative review:\n\n {state['review']}\n Return issue type, tone and urgency"""
    response = structured_model2.invoke(prompt)
    return {'diagnosis': response.model_dump()}

def negResponse(state: ReviewState) -> dict:
    # Safe access via get or ensuring state exists
    diagnosis = state.get('diagnosis') or {}
    
    prompt = f"""You are a support assistant. 
    The user had a {diagnosis.get('issue_type', 'unknown issue')}, 
    sounded {diagnosis.get('tone', 'concerned')} 
    and marked urgency as {diagnosis.get('urgency', 'medium')}. 
    Write an empathetic, helpful resolution message."""
    
    response = model.invoke(prompt).content
    return {'response': response}

In [39]:
# def findSentiment(state: ReviewState):
#     prompt = f'For the following review find out the sentiment \n {state["review"]}'
#     sentiment = structured_model.invoke(prompt).sentiment
#     return {'sentiment': sentiment}

# def checkSentiment(state: ReviewState) -> Literal["positiveResponse", "runDiagnosis"]:
#     if state.get('sentiment'):
#         return "positiveResponse"
#     else:
#         return "runDiagnosis"


# def positiveResponse(state: ReviewState):
#     prompt = f"""Write a warm Thank-you message in respose to this review:\n\n{state['review']}
#       Also Kindly ask the user to leave feedback to our website"""
#     response = model.invoke(prompt).content
#     return {'response': response}

# def runDiagnosis(state: ReviewState):
#     prompt = f"""Diagnose this negative review:\n\n {state['review']}\n
#       Return issue type, tone and urgency"""
#     response = structured_model2.invoke(prompt)

#     return {'diagnosis': response.model_dump()}

# def negResponse(state: ReviewState): 
    
#     diagnosis = state['diagnosis']
#     prompt = f"""You are a support assistant 
#     The user had a {diagnosis['issue_type']}, sounded {diagnosis['tone']} 
#      and marked urgency as {diagnosis['urgency']} Write an empathatic, helpfull resolution message"""
#     response = model.invoke(prompt).content

#     return {'response': response}





    



In [40]:

# Initialize your graph with the state
graph = StateGraph(ReviewState)

# Add all nodes
graph.add_node('findSentiment', findSentiment)
graph.add_node('positiveResponse', positiveResponse)
graph.add_node('runDiagnosis', runDiagnosis)
graph.add_node('negResponse', negResponse)

# Define standard flow edges
graph.add_edge(START, 'findSentiment')
graph.add_edge('positiveResponse', END)
graph.add_edge('runDiagnosis', 'negResponse')
graph.add_edge('negResponse', END)

# FIX: Add the conditional edge with an explicit routing map
graph.add_conditional_edges(
    'findSentiment',      # Source node
    checkSentiment,       # The routing function (do not put quotes around it!)
    {                     # Map: { router_output : target_node }
        "positiveResponse": "positiveResponse", 
        "runDiagnosis": "runDiagnosis"
    }
)

# Compile the workflow
workflow = graph.compile()


In [41]:
intState = {'review' : 'The product was really bad'}

fState = workflow.invoke(intState)

print(fState)

{'review': 'The product was really bad', 'sentiment': 'negative', 'diagnosis': {'issue_type': 'Other', 'tone': 'frustrated', 'urgency': 'medium'}, 'response': "Dear [User's Name],\n\nI understand how frustrating it can be to encounter issues, and I appreciate you reaching out for assistance. I'm here to help you resolve this matter as quickly and efficiently as possible.\n\nCould you please provide me with more details about the issue you're facing? Any specific information or examples will really help me get to the bottom of this for you. \n\nThank you for your patience, and I look forward to assisting you further!\n\nBest regards,  \n[Your Name]  \n[Your Title/Position]  \n[Your Contact Information]  \n"}
